# Notebook 02 — AH-PC Model: Architecture Definition and Sanity Checks

This notebook defines the full **Adaptive Horizon Predictive Coding (AH-PC)** model used in this MRP.

AH-PC builds on top of Sheibani et al.'s CNN predictive coding architecture for EEG. The core idea of
**predictive coding** is that a good brain signal representation should be one that can predict *future*
brain states — not just reconstruct the present. AH-PC extends this by predicting across *multiple time
horizons simultaneously*, with a learned attention mechanism that decides which horizon to emphasize.

## What this notebook does

1. Reproduces Sheibani's attention modules and CNN encoder **exactly** as in her original notebook
2. Defines the new AH-PC components: the **Horizon Attention Module** and **4 Decoder Heads**
3. Assembles everything into a single `AHPCModel` class
4. Prints a parameter count breakdown by component (including OLD vs NEW comparison)
5. Runs a forward pass on a dummy batch to confirm all output shapes
6. Verifies that horizon attention weights sum to 1.0 (sanity check on the softmax)

**No training happens here** — this notebook only defines and validates the architecture.

---

### Architecture overview

```
Input EEG segment: (B, 62 channels, 5 bands)
        │
        ▼
┌─────────────────────────────────────────────────────────┐
│  SHEIBANI ENCODER (reproduced exactly, untouched)       │
│  BandAttention → ChannelAttention → CNN → Bottleneck    │
│  Output: (B, 64, 15, 5)  ← spatial feature map          │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
     Projector (AH-PC): AdaptiveAvgPool2d((1,1)) → Flatten
                        (B, 64, 1, 1) → (B, 64)
                        Linear(64→256) + LayerNorm → z_t (B, 256)
                        │
           ┌────────────┴────────────┐
           ▼                         ▼
  Horizon Attention           4 × Decoder Heads
  α ∈ ℝ⁴, sums to 1          z̃_k ∈ ℝ²⁵⁶  for k ∈ {1,2,4,8}
```

## Setup and Imports

We only need PyTorch here — no dataset loading, no training utilities. The goal is to define the model
and run a quick sanity check using random (dummy) data.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

PyTorch version: 2.12.1+cu130
CUDA available:  False


---
## 1. Sheibani's Attention Modules: BandAttention and ChannelAttention

Each EEG segment has shape **(62 channels × 5 frequency bands)**. Before the CNN sees this 2D array,
Sheibani's model applies two lightweight attention gates in sequence:

**BandAttention** — asks: *which frequency bands matter right now?*
It pools information across all 62 channels (capturing a global view of each band), then learns
a multiplicative gate (0–1) for each of the 5 bands (Delta, Theta, Alpha, Beta, Gamma).

**ChannelAttention** — asks: *which brain regions (electrode channels) matter right now?*
It pools information across all 5 bands, then learns a gate for each of the 62 channels.

Both modules use the same design pattern — called **squeeze-and-excitation** — popularized in image
recognition: pool to get a summary → small MLP → sigmoid to produce a gate → multiply gate back
into the input. This is cheap to compute but lets the model dynamically suppress irrelevant features.

> **These two classes are copied verbatim from Sheibani's notebook** (cell `249a086d`).

In [11]:
class BandAttention(nn.Module):
    def __init__(self, num_bands):
        super().__init__()
        self.fc1 = nn.Linear(num_bands * 2, num_bands // 2)
        self.fc2 = nn.Linear(num_bands // 2, num_bands)

    def forward(self, x):  # x: (B, C, B_bands)
        avg_pool = x.mean(dim=1)            # (B, B_bands) — avg across channels
        max_pool = x.max(dim=1).values      # (B, B_bands) — max across channels
        context = torch.cat([avg_pool, max_pool], dim=-1)  # (B, 2*B_bands)
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))  # (B, B_bands)
        return x * weights.unsqueeze(1)     # broadcast gate over channel dim


class ChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.fc1 = nn.Linear(num_channels * 2, num_channels // 2)
        self.fc2 = nn.Linear(num_channels // 2, num_channels)

    def forward(self, x):  # x: (B, C, B_bands)
        avg_pool = x.mean(dim=2)            # (B, C) — avg across bands
        max_pool = x.max(dim=2).values      # (B, C) — max across bands
        context = torch.cat([avg_pool, max_pool], dim=-1)  # (B, 2*C)
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))  # (B, C)
        return x * weights.unsqueeze(2)     # broadcast gate over band dim


# Quick shape check
_x = torch.randn(4, 62, 5)
_ba = BandAttention(num_bands=5)
_ca = ChannelAttention(num_channels=62)
print("BandAttention    input:", _x.shape, "→ output:", _ba(_x).shape)
print("ChannelAttention input:", _x.shape, "→ output:", _ca(_x).shape)

BandAttention    input: torch.Size([4, 62, 5]) → output: torch.Size([4, 62, 5])
ChannelAttention input: torch.Size([4, 62, 5]) → output: torch.Size([4, 62, 5])


---
## 2. Sheibani's CNN Encoder and Bottleneck

After the two attention gates re-weight the input, the 2D EEG array is treated as a single-channel
image — think of it like a small grayscale image with 62 "pixels" of height and 5 "pixels" of width —
and passed through a CNN.

**Encoder block (two stages):**
- Stage 1: `Conv2d(1→16)` with 3×3 kernel → `BatchNorm` → `LeakyReLU(0.1)` → `MaxPool2d((2,1))`
  Reduces channel (height) dimension: 62 → 31. Band dimension (width) stays at 5.
- Stage 2: `Conv2d(16→32)` → `BatchNorm` → `ReLU` → `MaxPool2d((2,1))`
  Reduces channel dimension: 31 → 15.

**Bottleneck:** `Conv2d(32→64)` → `BatchNorm` → `ReLU`
  Output shape: **(B, 64 feature maps, 15, 5)**

The `SheibaniEncoder` class returns this raw spatial feature map — **no flattening**. The AH-PC
projector (Section 5) then applies `AdaptiveAvgPool2d((1,1))` to collapse the spatial dimensions
to `(B, 64, 1, 1)` → flatten to `(B, 64)` → `Linear(64→256)`. This keeps the projection cheap
and lets the convolutional layers do the spatial summarisation rather than a large dense layer.

> **The encoder and bottleneck are copied verbatim from Sheibani's notebook** (cells `0045da19`
> and `c752d4dc`). The reconstruction decoder from her original model is omitted because AH-PC
> predicts in latent space rather than reconstructing raw EEG.

In [12]:
class SheibaniEncoder(nn.Module):
    """
    Sheibani's attention + CNN encoder, reproduced exactly.
    Returns the raw bottleneck feature map (B, 64, 15, 5) — no flattening.
    Omits the ConvTranspose reconstruction decoder (not used in AH-PC).
    """

    OUT_CHANNELS = 64  # bottleneck output channels; projector reads this

    def __init__(self):
        super().__init__()

        self.band_attn = BandAttention(num_bands=5)
        self.channel_attn = ChannelAttention(num_channels=62)

        # Encoder: input (B, 1, 62, 5)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # (B, 16, 62, 5)
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d((2, 1)),                          # (B, 16, 31, 5)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # (B, 32, 31, 5)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),                          # (B, 32, 15, 5)
        )

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (B, 64, 15, 5)
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

    def forward(self, x):  # x: (B, 62, 5)
        x = self.band_attn(x)    # (B, 62, 5)  — band-gated
        x = self.channel_attn(x) # (B, 62, 5)  — channel-gated
        x = x.unsqueeze(1)       # (B, 1, 62, 5) — add channel dim for CNN
        x = self.encoder(x)      # (B, 32, 15, 5)
        x = self.bottleneck(x)   # (B, 64, 15, 5)
        return x                  # spatial feature map; projector handles pooling


# Shape check
_enc = SheibaniEncoder()
_out = _enc(torch.randn(4, 62, 5))
print(f"SheibaniEncoder input:  (4, 62, 5)")
print(f"SheibaniEncoder output: {_out.shape}  ← expected (4, 64, 15, 5)")

SheibaniEncoder input:  (4, 62, 5)
SheibaniEncoder output: torch.Size([4, 64, 15, 5])  ← expected (4, 64, 15, 5)


---
## 3. AH-PC Novel Contribution: Horizon Attention Module

This is the first component that is **new in AH-PC**. It takes the anchor embedding z_t (256-dim)
and produces four scalar weights — one per prediction horizon:

$$\alpha_t = \text{Softmax}(f_{\theta}(z_t)) \in \mathbb{R}^4, \quad \sum_k \alpha_{t,k} = 1$$

The four weights correspond to horizons k ∈ {1, 2, 4, 8} seconds ahead. Because they come from
a softmax, they form a probability distribution — the model is learning to say, for this particular
anchor segment, how much to weight each prediction horizon.

**Architecture:** `Linear(256→64)` → `LayerNorm(64)` → `GELU` → `Linear(64→4)` → `Softmax`

The hidden dimension is **64** (down from the original 128) to keep the module lightweight —
this is a routing/gating layer, not a feature extractor, so it does not need large capacity.

- **LayerNorm** (not BatchNorm): normalizes each sample's vector independently, appropriate
  for a 1D per-sample vector rather than a spatial feature map.
- **GELU**: a smooth activation function common in attention/transformer modules. It avoids the
  hard zero cutoff of ReLU, which can help gradient flow in small networks.
- The softmax is applied *outside* the Sequential so it is explicit and easy to inspect.

In [13]:
class HorizonAttentionModule(nn.Module):
    """
    Produces a soft probability distribution over K horizons given the anchor embedding.
    Output α sums to 1.0 across horizons (enforced by softmax).
    Hidden dim = 64 (lightweight routing layer, not a feature extractor).
    """

    def __init__(self, latent_dim=256, num_horizons=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, num_horizons),
        )

    def forward(self, z):  # z: (B, latent_dim)
        return F.softmax(self.net(z), dim=-1)  # (B, num_horizons)


# Shape check
_ham = HorizonAttentionModule(latent_dim=256, num_horizons=4)
_z = torch.randn(4, 256)
_alpha = _ham(_z)
print(f"HorizonAttentionModule input:  {_z.shape}")
print(f"HorizonAttentionModule output: {_alpha.shape}  ← expected (4, 4)")
print(f"Row sums (should all be 1.0):  {_alpha.sum(dim=-1).detach().numpy().round(5)}")

HorizonAttentionModule input:  torch.Size([4, 256])
HorizonAttentionModule output: torch.Size([4, 4])  ← expected (4, 4)
Row sums (should all be 1.0):  [1. 1. 1. 1.]


---
## 4. AH-PC Novel Contribution: Multi-Horizon Decoder Heads

AH-PC has **four independent decoder heads**, one for each prediction horizon k ∈ {1, 2, 4, 8}.
Each decoder takes the anchor embedding z_t (256-dim) and predicts what the bottleneck embedding
of the *target segment* k seconds later should look like:

$$\hat{z}_{t+k} = D_k(z_t), \quad k \in \{1, 2, 4, 8\}$$

By predicting in **latent space** (rather than reconstructing raw EEG), the model learns representations
that are *predictively meaningful* — the encoder is pushed to extract features that reveal where the
brain is heading, not just what it looks like right now. This is the core predictive coding principle.

The four decoders are **independent** (separate weights) because a 1-second and an 8-second prediction
likely require very different inductive biases.

**Architecture (same for all 4 horizons):**
`Linear(256→128)` → `ReLU` → `Dropout(0.1)` → `Linear(128→256)`

The hidden dimension is **128** (reduced from 512) to keep each decoder head compact. Because the
decoder only needs to transform one 256-dim vector into another, a narrow hidden layer is sufficient —
the convolutional encoder already carries the representational burden.

In [14]:
class HorizonDecoder(nn.Module):
    """
    Single-horizon decoder: maps anchor embedding z_t to a predicted target embedding z̃_{t+k}.
    One of these is instantiated for each prediction horizon.
    Hidden dim = 128 (compact; encoder carries the representational burden).
    """

    def __init__(self, latent_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, latent_dim),
        )

    def forward(self, z):  # z: (B, latent_dim)
        return self.net(z)  # (B, latent_dim)


# Shape check: four independent decoders
HORIZONS = [1, 2, 4, 8]
_decoders = nn.ModuleList([HorizonDecoder(latent_dim=256) for _ in HORIZONS])
_z = torch.randn(4, 256)
for k, dec in zip(HORIZONS, _decoders):
    _out = dec(_z)
    print(f"Decoder k={k:2d}: input {_z.shape} → output {_out.shape}")

Decoder k= 1: input torch.Size([4, 256]) → output torch.Size([4, 256])
Decoder k= 2: input torch.Size([4, 256]) → output torch.Size([4, 256])
Decoder k= 4: input torch.Size([4, 256]) → output torch.Size([4, 256])
Decoder k= 8: input torch.Size([4, 256]) → output torch.Size([4, 256])


---
## 5. The Complete AH-PC Model

The `AHPCModel` class wires everything together into a single PyTorch module:

| Step | Component | Input → Output |
|------|-----------|----------------|
| 1 | **Sheibani Encoder** (attention + CNN, untouched) | (B, 62, 5) → (B, 64, 15, 5) |
| 2 | **Projector** (Pool → Flatten → Linear + LN) | (B, 64, 15, 5) → (B, 64) → z_t (B, 256) |
| 3a | **Horizon Attention Module** | z_t → α (B, 4), sums to 1 |
| 3b | **4 × Decoder Heads** | z_t → {k: ẑ_{t+k} (B, 256)} |

The **Projector** (step 2) is a new component specific to AH-PC. Rather than flattening the full
(B, 64, 15, 5) tensor (which would give 4,800 inputs to a dense layer), it first applies
`AdaptiveAvgPool2d((1,1))` to collapse the 15×5 spatial grid into a single 64-dim channel summary.
This lets the convolutional layers do the spatial averaging naturally, and the linear layer only
needs to map 64 → 256 — dramatically reducing parameter count.

`LayerNorm` after the linear layer stabilises the distribution of z_t before it is consumed by
two different downstream modules (horizon attention and decoders).

**Two public methods:**
- `forward(anchor)` — full forward pass; returns `(z_t, alpha, predicted_embeddings)`
- `get_embedding(x)` — returns only `z_t`; used later during fine-tuning and evaluation

In [15]:
HORIZONS = [1, 2, 4, 8]
LATENT_DIM = 256


class AHPCModel(nn.Module):
    """
    Adaptive Horizon Predictive Coding (AH-PC) model.

    Extends Sheibani's CNN encoder (untouched) with:
    - A lightweight projector: AdaptiveAvgPool2d → Flatten → Linear(64→256) + LayerNorm
    - A Horizon Attention Module that produces a soft distribution over K horizons
    - K independent decoder heads that predict latent representations of future segments
    """

    def __init__(self, latent_dim: int = LATENT_DIM, horizons: list = HORIZONS):
        super().__init__()
        self.horizons = horizons

        # --- Sheibani's encoder (reproduced exactly, untouched) ---
        self.sheibani_encoder = SheibaniEncoder()

        # --- Projector: pool spatial dims → compact linear (new in AH-PC) ---
        # AdaptiveAvgPool2d collapses (B, 64, 15, 5) → (B, 64, 1, 1) with zero params.
        # Flatten gives (B, 64). Linear then maps to latent_dim.
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),      # (B, 64, 15, 5) → (B, 64, 1, 1)
            nn.Flatten(),                       # (B, 64, 1, 1)  → (B, 64)
            nn.Linear(SheibaniEncoder.OUT_CHANNELS, latent_dim),  # (B, 256)
            nn.LayerNorm(latent_dim),           # stabilise before branching
        )

        # --- Novel AH-PC components ---
        self.horizon_attn = HorizonAttentionModule(
            latent_dim=latent_dim,
            num_horizons=len(horizons),
        )
        self.decoders = nn.ModuleList([
            HorizonDecoder(latent_dim=latent_dim) for _ in horizons
        ])

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Return the 256-dim anchor embedding z_t.  Used during fine-tuning."""
        feat = self.sheibani_encoder(x)  # (B, 64, 15, 5)
        return self.projector(feat)       # (B, 256)

    def forward(self, anchor: torch.Tensor):
        """
        Args:
            anchor: (B, 62, 5)  — current EEG segment

        Returns:
            z_t:                  (B, 256)  — anchor latent embedding
            alpha:                (B, 4)    — horizon attention weights, sums to 1
            predicted_embeddings: dict {k: (B, 256)} for k in self.horizons
        """
        z_t = self.get_embedding(anchor)                           # (B, 256)
        alpha = self.horizon_attn(z_t)                             # (B, 4)
        predicted_embeddings = {
            k: self.decoders[i](z_t)
            for i, k in enumerate(self.horizons)
        }
        return z_t, alpha, predicted_embeddings


print("AHPCModel class defined successfully.")

AHPCModel class defined successfully.


---
## 6. Model Parameter Count Summary

Before any training, it is useful to understand how many learnable parameters each part of the model
contains. This tells us:
- How much of the model's capacity comes from Sheibani's original design vs. the new AH-PC additions
- Which components to freeze or fine-tune later
- Whether the model is reasonably sized for the dataset (~23,000 training pairs)

In [16]:
def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


model = AHPCModel(latent_dim=LATENT_DIM, horizons=HORIZONS)
enc = model.sheibani_encoder

# --- Sheibani sub-components (unchanged) ---
n_band      = count_params(enc.band_attn)
n_channel   = count_params(enc.channel_attn)
n_cnn       = count_params(enc.encoder)
n_bottle    = count_params(enc.bottleneck)
n_enc_total = count_params(enc)

# --- AH-PC additions (new design) ---
n_proj      = count_params(model.projector)
n_horizon   = count_params(model.horizon_attn)
n_decoders  = count_params(model.decoders)
n_total     = count_params(model)
ahpc_only   = n_proj + n_horizon + n_decoders

# --- Known OLD param counts (from original dense-projector design) ---
OLD = {
    "projector":  1_229_568,   # Linear(4800→256) + LayerNorm(256)
    "horizon":       33_668,   # Linear(256→128) + LN(128) + Linear(128→4)
    "decoders":   1_051_648,   # 4 × Linear(256→512) + Linear(512→256)
}
OLD["ahpc"]  = sum(OLD.values())
OLD["total"] = OLD["ahpc"] + n_enc_total

W = 44
print(f"{'Component':<{W}} {'OLD':>12} {'NEW':>12} {'Δ':>10}")
print("=" * (W + 38))
print(f"{'Sheibani Encoder (total, unchanged)':<{W}} {n_enc_total:>12,} {n_enc_total:>12,} {'—':>10}")
print(f"{'  ├─ BandAttention':<{W}} {n_band:>12,} {n_band:>12,} {'—':>10}")
print(f"{'  ├─ ChannelAttention':<{W}} {n_channel:>12,} {n_channel:>12,} {'—':>10}")
print(f"{'  ├─ CNN Encoder (2 conv stages)':<{W}} {n_cnn:>12,} {n_cnn:>12,} {'—':>10}")
print(f"{'  └─ Bottleneck (conv 32→64)':<{W}} {n_bottle:>12,} {n_bottle:>12,} {'—':>10}")
print("-" * (W + 38))

def delta(old, new):
    return f"{(new - old) / old * 100:+.0f}%"

print(f"{'Projector':<{W}} {OLD['projector']:>12,} {n_proj:>12,} {delta(OLD['projector'], n_proj):>10}")
print(f"{'  OLD: Linear(4800→256) + LayerNorm':<{W}}")
print(f"{'  NEW: AvgPool + Flatten + Linear(64→256) + LN':<{W}}")
print(f"{'Horizon Attention Module':<{W}} {OLD['horizon']:>12,} {n_horizon:>12,} {delta(OLD['horizon'], n_horizon):>10}")
print(f"{'  OLD: Linear(256→128) + LN + Linear(128→4)':<{W}}")
print(f"{'  NEW: Linear(256→64)  + LN + Linear(64→4)':<{W}}")
print(f"{'Decoder Heads (×4)':<{W}} {OLD['decoders']:>12,} {n_decoders:>12,} {delta(OLD['decoders'], n_decoders):>10}")
print(f"{'  OLD: each Linear(256→512) + Linear(512→256)':<{W}}")
print(f"{'  NEW: each Linear(256→128) + Linear(128→256)':<{W}}")
print("=" * (W + 38))
print(f"{'AH-PC novel params total':<{W}} {OLD['ahpc']:>12,} {ahpc_only:>12,} {delta(OLD['ahpc'], ahpc_only):>10}")
print(f"{'GRAND TOTAL':<{W}} {OLD['total']:>12,} {n_total:>12,} {delta(OLD['total'], n_total):>10}")
print()
print(f"Novel / Total ratio:  OLD = {OLD['ahpc']/OLD['total']:.1%}   NEW = {ahpc_only/n_total:.1%}")
print(f"Sheibani / Total ratio: OLD = {n_enc_total/OLD['total']:.1%}   NEW = {n_enc_total/n_total:.1%}")

Component                                             OLD          NEW          Δ
Sheibani Encoder (total, unchanged)                29,416       29,416          —
  ├─ BandAttention                                     37           37          —
  ├─ ChannelAttention                               5,859        5,859          —
  ├─ CNN Encoder (2 conv stages)                    4,896        4,896          —
  └─ Bottleneck (conv 32→64)                       18,624       18,624          —
----------------------------------------------------------------------------------
Projector                                       1,229,568       17,152       -99%
  OLD: Linear(4800→256) + LayerNorm         
  NEW: AvgPool + Flatten + Linear(64→256) + LN
Horizon Attention Module                           33,668       16,836       -50%
  OLD: Linear(256→128) + LN + Linear(128→4) 
  NEW: Linear(256→64)  + LN + Linear(64→4)  
Decoder Heads (×4)                              1,051,648      263,680       -7

---
## 7. Forward Pass Sanity Check

We now run the model on a **randomly generated dummy batch** to verify that all tensor shapes flow
through the network correctly. The dummy batch has:
- **B = 4** samples (batch size)
- Each sample has shape **(62, 5)** — matching our SEED-V processed data format

We run in `torch.no_grad()` mode since we are not computing gradients (no training here).

Expected output shapes:
- `z_t`:   `(4, 256)` — anchor embedding
- `alpha`: `(4, 4)`   — horizon attention weights
- `predicted_embeddings[k]`: `(4, 256)` for each k ∈ {1, 2, 4, 8}

In [17]:
torch.manual_seed(42)
B = 4
dummy_anchor = torch.randn(B, 62, 5)

model.eval()
with torch.no_grad():
    z_t, alpha, predicted_embeddings = model(dummy_anchor)

print("=" * 55)
print("Forward Pass Output Shapes")
print("=" * 55)
print(f"Input anchor:           {dummy_anchor.shape}")
print()
print(f"z_t  (anchor embedding):  {z_t.shape}")
print(f"     expected:            (4, 256)")
print()
print(f"alpha (horizon weights):  {alpha.shape}")
print(f"     expected:            (4, 4)")
print()
print("Predicted target embeddings:")
for k, z_hat in predicted_embeddings.items():
    print(f"  horizon k={k:2d}:  {z_hat.shape}   expected (4, 256)")

# Assert shapes
assert z_t.shape == (B, LATENT_DIM),        f"z_t shape mismatch: {z_t.shape}"
assert alpha.shape == (B, len(HORIZONS)),   f"alpha shape mismatch: {alpha.shape}"
for k in HORIZONS:
    assert predicted_embeddings[k].shape == (B, LATENT_DIM), \
        f"z_hat[{k}] shape mismatch: {predicted_embeddings[k].shape}"
print()
print("All output shapes are correct.")

Forward Pass Output Shapes
Input anchor:           torch.Size([4, 62, 5])

z_t  (anchor embedding):  torch.Size([4, 256])
     expected:            (4, 256)

alpha (horizon weights):  torch.Size([4, 4])
     expected:            (4, 4)

Predicted target embeddings:
  horizon k= 1:  torch.Size([4, 256])   expected (4, 256)
  horizon k= 2:  torch.Size([4, 256])   expected (4, 256)
  horizon k= 4:  torch.Size([4, 256])   expected (4, 256)
  horizon k= 8:  torch.Size([4, 256])   expected (4, 256)

All output shapes are correct.


---
## 8. Verifying Horizon Attention Weights Sum to 1.0

The `HorizonAttentionModule` ends with a **softmax**, which mathematically guarantees that the four
attention weights sum to exactly 1 for every sample. This is important because:

1. It lets the weights be interpreted as a probability distribution over horizons
2. The weighted sum of decoder losses (used later during training) stays on a consistent scale

The cell below checks this numerically for every sample in the dummy batch.

In [18]:
alpha_sums = alpha.sum(dim=-1)

print("Horizon attention weights per sample:")
print(f"  {'Sample':<10} {'k=1':>8} {'k=2':>8} {'k=4':>8} {'k=8':>8} {'sum':>10}")
print("  " + "-" * 55)
for i in range(B):
    w = alpha[i].detach().numpy()
    s = alpha_sums[i].item()
    print(f"  {i:<10} {w[0]:>8.4f} {w[1]:>8.4f} {w[2]:>8.4f} {w[3]:>8.4f} {s:>10.6f}")

assert torch.allclose(alpha_sums, torch.ones(B), atol=1e-5), \
    "Softmax invariant violated — alpha weights do not sum to 1!"
print()
print("All alpha weights sum to 1.0 (softmax invariant confirmed).")

Horizon attention weights per sample:
  Sample          k=1      k=2      k=4      k=8        sum
  -------------------------------------------------------
  0            0.2057   0.1989   0.3622   0.2331   1.000000
  1            0.2054   0.1986   0.3630   0.2330   1.000000
  2            0.2051   0.1986   0.3625   0.2338   1.000000
  3            0.2054   0.1987   0.3624   0.2336   1.000000

All alpha weights sum to 1.0 (softmax invariant confirmed).
